In [ ]:
# 1. Target Encoding (Multiclass & Binary)
import pandas as pd
df_final = pd.read_csv("./data/intermi/final_chess_dataset.csv")
multiclass_map = {"1-0": 2, "White": 2, "1/2-1/2": 1, "Draw": 1, "0-1": 0, "Black": 0}
binary_map = {"1-0": 1.0, "White": 1.0, "0-1": 0.0, "Black": 0.0}

df_final["winner_multiclass"] = df_final["result"].map(multiclass_map)
df_final["winner_binary"] = df_final["result"].map(binary_map)

# 2. Ordinal Encoding for Elo Buckets
ordinal_map = {"Beginner": 0, "Intermediate": 1, "Advanced": 2, "Expert": 3, "Master": 4}
df_final["elo_bucket_white_enc"] = df_final["elo_bucket_white"].map(ordinal_map)
df_final["elo_bucket_black_enc"] = df_final["elo_bucket_black"].map(ordinal_map)

print("Categorical features encoded successfully.")

In [ ]:
# Feature Engineering Math
df_final["elo_gap"] = df_final["white_elo"] - df_final["black_elo"]
df_final["avg_elo"] = ((df_final["white_elo"] + df_final["black_elo"]) / 2).round(0)
df_final["acl_gap"] = (df_final["white_acl"] - df_final["black_acl"]).round(2)

print("Engineered features added:")
print(df_final[['elo_gap', 'avg_elo', 'acl_gap']].head())

In [ ]:
# Drop raw text sequences
cols_to_drop = ["moves_pgn", "moves_uci", "moves_san"]
df_model_ready = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns])

print("=" * 50)
print("AFTER TRANSFORMATION SUMMARY")
print("=" * 50)
print(f"Final Model-Ready Shape: {df_model_ready.shape[0]:,} rows × {df_model_ready.shape[1]} columns")
print(f"Features dropped: {cols_to_drop}")

# Save the final dataset
df_model_ready.to_csv("./data/processed/merged_games_final.csv", index=False)
print("Dataset saved and ready for ML pipeline.")

### 4. Data Split & Balancing Strategy (Deferred to ML Pipeline)
* **Train/Validation/Test Split:** Will be executed using `sklearn.model_selection.train_test_split` (e.g., 80/10/10) directly inside the modeling script. This ensures features are scaled strictly on the training distribution to prevent data leakage.
* **Data Balancing Strategy:** Based on our validation report, our target (`winner_multiclass`) is naturally imbalanced: White wins (~18.9k), Black wins (~16k), and Draws (~8.7k)[cite: 1]. Since predicting draws is notoriously difficult in chess, we will utilize class weighting (e.g., `class_weight='balanced'` in sklearn) or SMOTE during the model training phase to prevent the algorithm from ignoring the minority 'Draw' class.